# EDA v2 для кейса №5: Доставка еды

**Задача:** регрессия `actual_delivery_time_minutes` для завершённых заказов.

Эта итерация проходится по EDA v1 и проводит следующие правки:

- убран широкий, общий анализ всех числовых признаков (деньги, ID, пост-фактум поля);
- центральный объект анализа — **остаток `actual - estimated`**, а не общие boxplots, исследуем разброс с предсказанием платформы;
- фокус на драйверах: расстояние × пробки × погода × время приготовления, временные пики;
- явный список **утечек** и рабочий набор признаков для модели;
- два сценария моделирования: **без** `estimated` и **с** `estimated` (или на остатке).

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pd.set_option('display.max_columns', None)
pio.templates.default = 'plotly_white'

## Часть 0. Загрузка, типизация, фильтр `Completed`

Пропуски в v1 уже показаны системными (у `Cancelled` нет таргета, у `Completed` нет причины отмены) — здесь не повторяем, сразу фиксируем фильтр. Дубликаты — быстрый санити-чек одной ячейкой.

In [2]:
df = pd.read_csv(r'Z:/Coding/URFU projects/Semester 1/Case 5 - Food delivery/Data/food_delivery_orders_dataset.csv')

cat_cols = ['city', 'delivery_area', 'customer_type', 'restaurant_type', 'restaurant_primary_category',
            'payment_method', 'weather', 'traffic_level', 'day_of_week', 'order_status', 'cancellation_reason']
for c in cat_cols:
    df[c] = df[c].astype('category')
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_timestamp'] = pd.to_datetime(df['order_timestamp'])

# Санити-чек дубликатов (без отдельного блока — по ревью)
print('Полных дубликатов строк:', df.duplicated().sum(),
      '| дубликатов order_id:', df['order_id'].duplicated().sum())

d = df[df['order_status'] == 'Completed'].copy()
print(f'Рабочий датафрейм d: {d.shape[0]} из {df.shape[0]} заказов ({(df["order_status"] == "Cancelled").mean():.2%} отмен отфильтровано)')
print(f'Период: {d["order_date"].min():%d.%m.%Y} — {d["order_date"].max():%d.%m.%Y}')

Полных дубликатов строк: 0 | дубликатов order_id: 0
Рабочий датафрейм d: 49132 из 50000 заказов (1.74% отмен отфильтровано)
Период: 01.10.2025 — 30.10.2025


## Часть 1. Утечки и рабочий список признаков

Для регрессии времени доставки не все столбцы допустимы. Классификация с обоснованием:

| Признак | Статус | Причина |
|---|---|---|
| `order_id`, `customer_id`, `restaurant_id`, `driver_id` | **исключить** | ID без закономерностей; для исторических агрегатов — только с time-based CV |
| `order_timestamp`, `order_date` | **исключить (сырые)** | высокая кардинальность; оставляем производные: `order_hour`, `day_of_week`, `is_weekend` |
| `tip_amount`, `customer_rating` | **утечка** | пост-фактум, известны после доставки |
| `late_delivery` | **утечка** | производная от `actual` и `estimated` |
| `cancellation_reason`, `order_status` | **исключить** | нужны только для фильтра; после фильтра константа/пропуск |
| `subtotal`, `discount_percent`, `tax_amount`, `service_fee`, `delivery_fee`, `order_total`, `payment_method` | **исключить** | деньги/платёж слабо связаны со временем и сильно коррелируют между собой |
| `customer_age`, `customer_type` | **исключить** | слабая связь с временем доставки, шум |
| `distance_km`, `weather`, `traffic_level`, `restaurant_preparation_time_minutes`, `delivery_partner_experience_months`, `delivery_partner_rating`, `order_hour`, `day_of_week`, `is_weekend`, `city`, `delivery_area` | **ядро** | доступны на момент заказа, физически влияют на время |
| `items_count`, `restaurant_type`, `restaurant_primary_category`, `restaurant_rating` | **проверить** | частично поглощаются `prep_time`; держим вне первой модели без необходимости |
| `estimated_delivery_time_minutes` | **особый статус** | доступен на момент заказа и очень силён, но в первой модели — только как baseline / объект анализа остатка, иначе съедает важности остальных признаков |

**`delivery_area` — 40 категорий**: проверяем размер групп, для модели — target encoding только внутри CV.

## Часть 2. Таргет и выбросы

Распределение, квантили, экстремумы и их правдоподобность.

In [3]:
t = d['actual_delivery_time_minutes']
print(t.describe().round(1).to_string())
print()
for q in [0.90, 0.95, 0.99]:
    print(f'p{int(q*100)}: {t.quantile(q):.1f} мин')
print(f'> 60 мин: {(t > 60).mean():.2%} | > 90 мин: {(t > 90).mean():.2%} | max: {t.max():.0f} мин')

fig = go.Figure()
fig.add_trace(go.Histogram(x=t, nbinsx=70, marker_color='#636EFA'))
fig.update_layout(height=400, title='Распределение фактического времени доставки (мин)',
                  template='plotly_white', bargap=0.02)
fig.show()

count    49132.0
mean        35.2
std         14.7
min         10.0
25%         25.0
50%         33.0
75%         42.0
max        172.0

p90: 54.0 мин
p95: 63.0 мин
p99: 85.0 мин
> 60 мин: 5.91% | > 90 мин: 0.64% | max: 172 мин


In [4]:
# Топ-10 самых долгих заказов: проверка правдоподобности экстремумов
extreme = d.nlargest(10, 'actual_delivery_time_minutes')[
    ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes', 'distance_km',
     'restaurant_preparation_time_minutes', 'weather', 'traffic_level']]
print(extreme.to_string(index=False))

 actual_delivery_time_minutes  estimated_delivery_time_minutes  distance_km  restaurant_preparation_time_minutes weather traffic_level
                        172.0                               97         19.7                                    7   Storm          High
                        158.0                               96         17.3                                   22   Storm          High
                        156.0                               88         16.9                                   11   Storm          High
                        153.0                               98         18.5                                   33   Storm        Medium
                        147.0                              109         19.0                                   24    Rain          High
                        144.0                               81         20.0                                   11   Storm        Medium
                        143.0                          

In [5]:
# Санити-чек скорости (перенос из v1, сокращённо)
d['transit_min'] = d['actual_delivery_time_minutes'] - d['restaurant_preparation_time_minutes']
ok = d[d['transit_min'] > 0].copy()
ok['speed_kmh'] = ok['distance_km'] / ok['transit_min'] * 60
print(f'Заказов с transit <= 0: {(d["transit_min"] <= 0).sum()} '
      f'({(d["transit_min"] <= 0).mean():.2%}) — артефакт, исключаем из анализа скорости')
print(f'Медианная скорость: {ok["speed_kmh"].median():.1f} км/ч, p95: {ok["speed_kmh"].quantile(0.95):.1f} км/ч')
print(ok.groupby('weather', observed=True)['speed_kmh'].median().round(1).to_string())

Заказов с transit <= 0: 74 (0.15%) — артефакт, исключаем из анализа скорости
Медианная скорость: 14.7 км/ч, p95: 25.3 км/ч
weather
Clear     15.8
Cloudy    14.4
Rain      12.0
Storm      9.3


**Выводы по таргету:** правый хвост (максимумы 140–172 мин) соответствует большим `distance_km` + `prep_time` в шторм/пробки — экстремумы физически правдоподобны, сохраняем. Хвост короткий (<2% выше 90 мин), поэтому:

- бизнес-метрики считаем в **исходной шкале** (минуты интерпретируемы);
- для модели сравним `MAE` на исходной и **log-шкале** (колонка `log_target` ниже); winsorize не применяем, чтобы не портить интерпретируемость.

In [6]:
d['log_target'] = np.log1p(d['actual_delivery_time_minutes'])
# остаток платформы — главный объект анализа
d['delay'] = d['actual_delivery_time_minutes'] - d['estimated_delivery_time_minutes']
print(f'Остаток delay = actual - estimated: mean {d["delay"].mean():+.2f}, median {d["delay"].median():+.2f}')
print(f'Доля опозданий (actual > estimated): {(d["actual_delivery_time_minutes"] > d["estimated_delivery_time_minutes"]).mean():.1%}')

Остаток delay = actual - estimated: mean +4.23, median +4.00
Доля опозданий (actual > estimated): 75.9%


## Часть 3. Главный блок: `actual` vs `estimated`

`estimated_delivery_time_minutes` известен на момент заказа, поэтому это не есть утечка, но в модели его место особое: это baseline платформы. Сначала анализируем его качество.

In [7]:
sample = d.sample(3000, random_state=42)

fig = px.scatter(sample, x='estimated_delivery_time_minutes', y='actual_delivery_time_minutes',
                 opacity=0.5, color_discrete_sequence=['#636EFA'], template='plotly_white', height=500,
                 title='Факт vs прогноз платформы (линия y=x — идеальный прогноз)',
                 labels={'estimated_delivery_time_minutes': 'Прогноз, мин',
                         'actual_delivery_time_minutes': 'Факт, мин'})
lim = [0, d['actual_delivery_time_minutes'].max() + 5]
fig.add_scatter(x=lim, y=lim, mode='lines', name='y = x',
                line=dict(color='#EF553B', dash='dash'), showlegend=True)
fig.update_xaxes(range=lim); fig.update_yaxes(range=lim)
fig.show()

In [8]:
print(f'Остаток delay: mean {d["delay"].mean():+.2f} мин, median {d["delay"].median():+.2f} мин, std {d["delay"].std():.1f}')
print(f'Доля опозданий (delay > 0): {(d["delay"] > 0).mean():.1%}')
print(f'Связь платформы с фактом r(estimated, actual) = '
      f'{d["estimated_delivery_time_minutes"].corr(d["actual_delivery_time_minutes"]):.3f}')

fig = go.Figure()
fig.add_trace(go.Histogram(x=d['delay'], nbinsx=80, marker_color='#EF553B'))
fig.add_vline(x=0, line_dash='dash', line_color='#636EFA',
              annotation_text='0 (прогноз точен)')
fig.update_layout(height=400, title='Остаток actual - estimated (мин): правый хвост = опоздания',
                  template='plotly_white', bargap=0.02)
fig.show()

Остаток delay: mean +4.23 мин, median +4.00 мин, std 5.5
Доля опозданий (delay > 0): 75.9%


Связь платформы с фактом r(estimated, actual) = 0.930


In [9]:
# Остаток по внешним условиям: погода, пробки, расстояние, час, город
weather_order = ['Clear', 'Cloudy', 'Rain', 'Storm']
traffic_order = ['Low', 'Medium', 'High']
d['dist_bin'] = pd.cut(d['distance_km'], [0, 2, 4, 6, 8, 10, 25],
                       labels=['0–2', '2–4', '4–6', '6–8', '8–10', '10+'])

for col, order in [('weather', weather_order), ('traffic_level', traffic_order),
                   ('dist_bin', None), ('order_hour', None), ('city', None)]:
    seg = d.groupby(col, observed=True).agg(
        n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
        mean_est=('estimated_delivery_time_minutes', 'mean'),
        delay=('delay', 'mean'), late_rate=('delay', lambda s: (s > 0).mean()))
    if order is not None:
        seg = seg.reindex(order)
    print(f'--- {col} ---')
    print(seg.round(2).to_string(), '\n')

--- weather ---
             n  mean_actual  mean_est  delay  late_rate
weather                                                
Clear    27857        33.56     31.01   2.55       0.68
Cloudy   12581        35.12     30.83   4.29       0.80
Rain      7427        39.16     30.66   8.50       0.94
Storm     1267        47.07     31.42  15.65       0.98 



--- traffic_level ---
                   n  mean_actual  mean_est  delay  late_rate
traffic_level                                                
Low            14987        30.59     26.72   3.88       0.75
Medium         19197        35.02     30.81   4.21       0.76
High           14948        39.89     35.27   4.62       0.76 

--- dist_bin ---
              n  mean_actual  mean_est  delay  late_rate
dist_bin                                                
0–2        5267        22.59     19.41   3.18       0.73
2–4       19149        28.41     24.77   3.64       0.75
4–6       13182        35.90     31.60   4.30       0.77
6–8        6198        43.83     38.86   4.98       0.78
8–10       2825        52.38     46.57   5.81       0.79
10+        2511        68.18     61.17   7.01       0.81 

--- order_hour ---
               n  mean_actual  mean_est  delay  late_rate
order_hour                                               
0            423        30.01     26.32   3.69       0.7

In [10]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Остаток по погоде', 'Остаток по пробкам'))
for i, (col, order) in enumerate([('weather', weather_order), ('traffic_level', traffic_order)]):
    seg = d.groupby(col, observed=True)['delay'].mean().reindex(order)
    fig.add_bar(x=seg.index.astype(str), y=seg.values, showlegend=False,
                marker_color='#EF553B' if i == 0 else '#636EFA', row=1, col=i+1)
fig.update_layout(height=420, title='Средний остаток (факт − прогноз), мин', template='plotly_white')
fig.show()

**Выводы:**

- Прогноз платформы в среднем **занижен на ~4 мин** (mean delay +4.23, median +4.0), и ошибка монотонно растёт с расстоянием: 3.2 мин на 0-2 км против 7.0 мин на 10+ км.
- Прогноз **«слеп» к погоде**: средний прогноз ~31 мин при любой погоде, тогда как факт растёт с 33.6 (Clear) до 47.1 мин (Storm); остаток - с +2.6 до +15.7 мин, а доля опозданий (факт > прогноз) — с 68% до 98%. По консервативному флагу late_delivery (v1) — 24% до 89%. Пробки Low→High добавляют ~9 мин факта.
- К часу суток и городам прогноз адаптирован - остаток там почти не меняется. Улучшение ETA нужно искать именно в погоде, пробках и дистанции.

## Часть 4. Ключевые драйверы времени доставки
### 4.1 Расстояние: бины, среднее/медиана, доля опозданий

In [11]:
segd = d.groupby('dist_bin', observed=True).agg(
    n=('delay', 'size'), mean=('actual_delivery_time_minutes', 'mean'),
    median=('actual_delivery_time_minutes', 'median'),
    late_rate=('delay', lambda s: (s > 0).mean()))
print(segd.round(2).to_string())

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=segd.index.astype(str), y=segd['mean'], name='среднее время, мин', marker_color='#636EFA')
fig.add_scatter(x=segd.index.astype(str), y=segd['late_rate'], name='доля опозданий',
                mode='lines+markers', secondary_y=True, line=dict(color='#EF553B', width=3))
fig.update_yaxes(title_text='Время, мин', secondary_y=False)
fig.update_yaxes(title_text='Доля опозданий', secondary_y=True)
fig.update_layout(height=420, title='Время доставки и опоздания по бинам расстояния',
                  template='plotly_white', showlegend=False)
fig.show()

              n   mean  median  late_rate
dist_bin                                 
0–2        5267  22.59    21.0       0.73
2–4       19149  28.41    27.0       0.75
4–6       13182  35.90    35.0       0.77
6–8        6198  43.83    43.0       0.78
8–10       2825  52.38    51.0       0.79
10+        2511  68.18    65.0       0.81


### 4.2 Время приготовления и взаимодействие с `items_count`

In [12]:
d['prep_bin'] = pd.cut(d['restaurant_preparation_time_minutes'], [0, 10, 20, 30, 40, 65],
                       labels=['0–10', '10–20', '20–30', '30–40', '40+'])
prep = d.groupby('prep_bin', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    late_rate=('delay', lambda s: (s > 0).mean()))
print(prep.round(2).to_string())

heat = d.pivot_table(index='prep_bin', columns=pd.cut(d['items_count'], [0, 2, 5, 15],
                     labels=['1–2', '3–5', '6+']), values='actual_delivery_time_minutes',
                     observed=True, aggfunc='mean')
fig = px.imshow(heat.round(1), text_auto=True, color_continuous_scale='RdYlGn_r', aspect='auto',
                title='Среднее время доставки (мин): время готовки × число позиций')
fig.update_layout(height=420, template='plotly_white')
fig.show()

              n  mean_actual  late_rate
prep_bin                               
0–10      12434        27.44       0.53
10–20     26466        34.50       0.83
20–30      8220        43.64       0.86
30–40      1611        54.13       0.87
40+         390        66.45       0.87


### 4.3 Взаимодействие `distance × traffic`

In [13]:
pv = d.pivot_table(index='dist_bin', columns='traffic_level', values='delay',
                   observed=True, aggfunc='mean').reindex(columns=traffic_order)
print('Средний остаток (факт − прогноз), мин:')
print(pv.round(2).to_string())

fig = px.imshow(pv.round(1), text_auto=True, color_continuous_scale='RdYlGn_r', aspect='auto',
                title='Остаток (мин): расстояние × уровень пробок')
fig.update_layout(height=420, template='plotly_white')
fig.show()

pva = d.pivot_table(index='dist_bin', columns='traffic_level', values='actual_delivery_time_minutes',
                    observed=True, aggfunc='mean').reindex(columns=traffic_order)
print('Средний факт, мин:')
print(pva.round(1).to_string())

Средний остаток (факт − прогноз), мин:
traffic_level   Low  Medium  High
dist_bin                         
0–2            3.05    3.18  3.32
2–4            3.42    3.63  3.88
4–6            3.94    4.24  4.73
6–8            4.57    4.92  5.46
8–10           5.00    5.98  6.43
10+            5.94    6.82  8.27


Средний факт, мин:
traffic_level   Low  Medium  High
dist_bin                         
0–2            21.1    22.6  24.1
2–4            25.6    28.3  31.4
4–6            31.3    35.6  41.0
6–8            37.3    43.6  50.7
8–10           43.6    52.5  61.2
10+            55.2    67.7  81.0


### 4.4 Курьеры: есть ли эффект после контроля расстояния и пробок

In [14]:
d['exp_bin'] = pd.cut(d['delivery_partner_experience_months'], [-1, 6, 18, 36, 61],
                      labels=['<6', '6–18', '18–36', '36+'])
d['rating_bin'] = pd.cut(d['delivery_partner_rating'], [0, 4.0, 4.4, 4.7, 5.0],
                         labels=['<4.0', '4.0–4.4', '4.4–4.7', '4.7+'])

# Контроль: смотрим только «средний» сегмент расстояния 2–6 км
mid = d[d['dist_bin'].isin(['2–4', '4–6'])]
for col in ['exp_bin', 'rating_bin']:
    seg = mid.groupby(col, observed=True).agg(
        n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
        late_rate=('delay', lambda s: (s > 0).mean()))
    print(f'--- {col} (контроль: расстояние 2–6 км) ---')
    print(seg.round(2).to_string(), '\n')

--- exp_bin (контроль: расстояние 2–6 км) ---
             n  mean_actual  late_rate
exp_bin                               
<6        3824        31.61       0.75
6–18      5372        31.35       0.75
18–36    11011        31.41       0.75
36+      12124        31.52       0.76 

--- rating_bin (контроль: расстояние 2–6 км) ---
                n  mean_actual  late_rate
rating_bin                               
<4.0         3745        31.89       0.76
4.0–4.4     10413        31.25       0.75
4.4–4.7      9144        31.41       0.76
4.7+         9029        31.59       0.75 



**Выводы по драйверам:**

- `distance_km` - главный внешний драйвер, эффект монотонный; среднее время 22.6 → 68.2 мин от 0-2 км до 10+ км, доля опозданий (факт > прогноз) растёт с 73% до 81%.
- `prep_time` сильный, взаимодействие с `items_count` даёт лишние минуты, но вклад умеренный - `items_count` оставляем «проверить», без гарантии попадания в модель.
- `distance × traffic` - существенное взаимодействие: на дальних плечах в пробках остаток ошибки прогноза максимален.
- Опыт и рейтинг курьера дают **слабый** эффект даже после контроля расстояния.

## Часть 5. Временные закономерности

In [15]:
segh = d.groupby('order_hour', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    mean_delay=('delay', 'mean'), late_rate=('delay', lambda s: (s > 0).mean())).reset_index()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=segh['order_hour'], y=segh['n'], name='заказов', marker_color='#B6B6F5')
fig.add_scatter(x=segh['order_hour'], y=segh['mean_actual'], name='среднее время, мин',
                mode='lines+markers', line=dict(color='#636EFA', width=3), secondary_y=True)
fig.update_yaxes(title_text='Заказов', secondary_y=False)
fig.update_yaxes(title_text='Среднее время, мин', secondary_y=True)
fig.update_layout(height=420, title='Часы суток: спрос и среднее время доставки', template='plotly_white')
fig.show()
print(segh.round(2).to_string(index=False))

 order_hour    n  mean_actual  mean_delay  late_rate
          0  423        30.01        3.69       0.73
          1  430        31.48        3.70       0.72
          2  405        31.81        3.68       0.75
          3  404        31.32        4.14       0.75
          4  422        31.51        4.09       0.76
          5  395        30.97        3.64       0.71
          6  852        33.17        4.21       0.79
          7 1288        34.17        3.89       0.73
          8 1647        33.86        4.24       0.77
          9 2069        34.13        4.22       0.75
         10 2572        33.46        4.02       0.75
         11 3381        33.60        4.13       0.76
         12 4129        36.65        4.36       0.76
         13 3763        37.12        4.36       0.76
         14 3045        37.27        4.45       0.76
         15 1585        34.06        4.26       0.76
         16 1697        33.90        4.17       0.76
         17 2086        33.64        4.20     

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
segw = d.groupby('day_of_week', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    late_rate=('delay', lambda s: (s > 0).mean())).reindex(dow_order)
print(segw.round(2).to_string())

# Ежедневная динамика за 30 дней: тренд/аномалии
daily = d.groupby('order_date', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    late_rate=('delay', lambda s: (s > 0).mean()))
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=daily.index, y=daily['n'], name='заказов/день', marker_color='#B6B6F5')
fig.add_scatter(x=daily.index, y=daily['mean_actual'], name='среднее время, мин',
                mode='lines+markers', line=dict(color='#636EFA', width=3), secondary_y=True)
fig.update_yaxes(title_text='Заказов/день', secondary_y=False)
fig.update_yaxes(title_text='Среднее время, мин', secondary_y=True)
fig.update_layout(height=420, title='Динамика по датам (октябрь 2025)', template='plotly_white')
fig.show()
print(f'Среднее время по дням: min {daily["mean_actual"].min():.1f}, max {daily["mean_actual"].max():.1f} мин')

                n  mean_actual  late_rate
day_of_week                              
Monday       6540        36.63       0.77
Tuesday      6558        35.15       0.77
Wednesday    8050        35.80       0.79
Thursday     8316        35.35       0.78
Friday       6517        34.62       0.74
Saturday     6595        34.55       0.74
Sunday       6556        33.76       0.71


Среднее время по дням: min 33.4, max 38.7 мин


**Выводы:**

- **Пики спроса** - обед (~12–13 ч) и вечер (19–21 ч); ночью заказов почти нет. Среднее время в пики чуть выше, но остаток платформы от часа почти не зависит — прогноз адаптирован к нагрузке (подтверждает v1).
- Всплеск Wed–Thu - **артефакт календаря** месяца, не поведение клиентов.
- По датам за 30 дней **выраженного тренда или аномалий нет** (среднее и late_rate колеблются в узком диапазоне) → допустим как time-based split (последние N дней в тест), так и случайный; для честности выбираем **time-based**.

## Часть 6. Категориальные признаки: `city`, `delivery_area`, тип ресторана

In [17]:
segc = d.groupby('city', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    late_rate=('delay', lambda s: (s > 0).mean()), mean_delay=('delay', 'mean'))
print(segc.round(2).to_string())

# delivery_area: 40 категорий — размер групп, top/bottom по времени (мелкие группы помечаем)
sega = d.groupby('delivery_area', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    late_rate=('delay', lambda s: (s > 0).mean()))
sega['low_n'] = sega['n'] < 100
print(f'\nРайонов: {sega.shape[0]}, с < 100 заказов (шумные оценки): {sega["low_n"].sum()}')
print('\nТоп-5 самых долгих районов:')
print(sega.nlargest(5, 'mean_actual').round(2).to_string())
print('\nТоп-5 самых быстрых районов:')
print(sega.nsmallest(5, 'mean_actual').round(2).to_string())

            n  mean_actual  late_rate  mean_delay
city                                             
City_A  19527        34.99       0.76        4.38
City_B  14627        36.01       0.76        4.52
City_C  10006        34.06       0.76        3.67
City_D   4972        35.47       0.73        3.94

Районов: 40, с < 100 заказов (шумные оценки): 0

Топ-5 самых долгих районов:
                   n  mean_actual  late_rate  low_n
delivery_area                                      
City_B_Area_8   1505        37.03       0.79  False
City_B_Area_3   1520        36.78       0.77  False
City_D_Area_8    512        36.22       0.76  False
City_B_Area_6   1480        36.14       0.77  False
City_B_Area_10  1421        36.14       0.79  False

Топ-5 самых быстрых районов:
                  n  mean_actual  late_rate  low_n
delivery_area                                     
City_C_Area_6  1014        33.50       0.76  False
City_C_Area_3   980        33.56       0.74  False
City_C_Area_9  1030     

In [18]:
# Тип и категория ресторана: сравниваем ВМЕСТЕ с prep_time — эффект может «поглощаться» готовкой
segr = d.groupby('restaurant_type', observed=True).agg(
    n=('delay', 'size'), mean_actual=('actual_delivery_time_minutes', 'mean'),
    mean_prep=('restaurant_preparation_time_minutes', 'mean'),
    mean_dist=('distance_km', 'mean'))
print('По типу ресторана (время + средние prep/dist для контроля):')
print(segr.round(2).to_string())

По типу ресторана (время + средние prep/dist для контроля):
                     n  mean_actual  mean_prep  mean_dist
restaurant_type                                          
Cafe              7999        31.29      11.85       4.73
Casual Dining    12848        38.60      18.97       4.73
Fast Food        27106        34.11      14.45       4.73
Fine Dining       1179        47.74      28.45       4.75


**Выводы:**

- `city` и `delivery_area` дают вариативность, но умеренную; у `delivery_area` (40 категорий) часть групп мала → в модели только **target encoding внутри CV** (или частотная агрегация), иначе утечка.
- Различия по `restaurant_type` почти целиком объясняются средним `prep_time` и `distance_km` — тип ресторана не делать главным драйвером.
- Для высококардинального `restaurant_primary_category` — тот же подход: target encoding только внутри CV.

## Часть 7. Корреляции — только «чистый» набор

По ревью: без ID, денег, `late_delivery`, `tip_amount`, `customer_rating`. `estimated` показываем отдельно — это baseline, а не обычная мультиколлинеарность.

In [ ]:
%pip install phik -q
import phik


[notice] A new release of pip is available: 23.3.2 -> 26.3
[notice] To update, run: C:\Users\Nikita\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [20]:
core_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
             'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
             'order_hour', 'delivery_partner_experience_months', 'delivery_partner_rating',
             'weather', 'traffic_level', 'city', 'delivery_area', 'restaurant_type',
             'restaurant_primary_category', 'is_weekend', 'day_of_week']
interval_cols = ['actual_delivery_time_minutes', 'estimated_delivery_time_minutes',
                 'restaurant_preparation_time_minutes', 'distance_km', 'items_count',
                 'order_hour', 'delivery_partner_experience_months', 'delivery_partner_rating']

phik_matrix = d.sample(10000, random_state=42)[core_cols].phik_matrix(interval_cols=interval_cols)

fig = px.imshow(phik_matrix.round(2), text_auto=True, aspect='auto',
                color_continuous_scale='RdBu_r', zmin=0, zmax=1,
                title='Phik-матрица: только релевантные признаки (выборка 10k)')
fig.update_layout(height=750, template='plotly_white')
fig.show()

target_phik = phik_matrix['actual_delivery_time_minutes'].drop('actual_delivery_time_minutes').sort_values(ascending=False)
print('Топ связей с таргетом:')
print(target_phik.round(3).to_string())

Топ связей с таргетом:
estimated_delivery_time_minutes        0.918
distance_km                            0.790
restaurant_preparation_time_minutes    0.557
items_count                            0.415
traffic_level                          0.279
weather                                0.199
restaurant_type                        0.191
order_hour                             0.099
delivery_partner_experience_months     0.063
day_of_week                            0.057
is_weekend                             0.055
restaurant_primary_category            0.040
city                                   0.019
delivery_area                          0.019
delivery_partner_rating                0.000


**Выводы:**

- Таргет определяется **расстоянием, временем готовки и `estimated`** (последний - производная платформы от тех же величин, поэтому его связь с фактом близка к 1 - это не мультиколлинеарность, а «baseline-подобная» связь).
- Погода, пробки, город - слабые парные связи, но они объясняют **остаток и хвосты** (см. Части 3-4): парные корреляции их недооценивают.
- Отбор признаков в модели делаем не по парным корреляциям, а по **важностям модели / permutation importance / SHAP**, т.к. связи нелинейные и взаимодействуют.

## Часть 8. Итоги: рабочий набор признаков и план моделирования

**Ядро признаков (сценарий 1 — «без estimated»):**
`distance_km`, `weather`, `traffic_level`, `delivery_partner_experience_months`, `delivery_partner_rating`, `restaurant_preparation_time_minutes`, `order_hour`, `day_of_week`, `is_weekend`, `city`, `delivery_area` (target encoding в CV).

**Сценарий 2 — «с estimated»:** то же ядро + `estimated_delivery_time_minutes` как признак **или** моделирование остатка `actual - estimated`.

**Исключено:** все сырые ID, `order_timestamp`/`order_date` (кроме производных), `tip_amount`, `customer_rating`, `late_delivery`, `cancellation_reason`, `order_status`, все денежные/платёжные признаки, `customer_age`, `customer_type` — утечки или шум.

**Обработка таргета:** выбросы физически правдоподобны — сохраняем; сравнить MAE на исходной и log-шкале (`log_target`).

**CV:** time-based split (данные без тренда, но месяц короткий — честнее по времени); target encoding только внутри CV.

**Главные находки (согласуются с v1, но теперь с фокусом на остаток):**
1. Прогноз платформы в среднем занижен на ~4 мин; 76% доставок позже прогноза (по флагу late_delivery — 37%).
2. Ошибка растёт с расстоянием (6–7 мин на 10+ км) и «слепа» к погоде (89% опозданий в Storm).
3. Драйверы: distance ≫ prep_time > traffic/weather; курьеры почти не влияют.
4. Пики и города платформа учитывает — улучшать нужно погоду, пробки, дистанцию.